In [1]:
import os
import boto3
import duckdb
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from deltalake import DeltaTable, write_deltalake
import polars as pl
import json
# Load .env file
load_dotenv()

# Read environment variables
endpoint = os.getenv("MINIO_ENDPOINT")
access_key = os.getenv("MINIO_ACCESS_KEY")
secret_key = os.getenv("MINIO_SECRET_KEY")

In [2]:
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint, # MinIO API endpoint
    aws_access_key_id=access_key, # User name
    aws_secret_access_key=secret_key, # Password
)

In [3]:
base_uri = "persistent-landing/semistructured/"

In [47]:
def get_deep_keys(data,level=0):
    # If it's a list, dive into the first element
    if isinstance(data, list) and len(data) > 0:
        return get_deep_keys(data[0],level+1)

    # If it's finally a dictionary, return the keys
    if isinstance(data, dict):
        return list(data.keys()),level

    # If it's a primitive (like a string or number) or empty
    return [],level

def extract_timestamp_from_filename(filename):
    # Strip extension and split by underscore
    name_part = os.path.splitext(filename)[0]
    raw_ts = name_part.split('_')[-1]

    try:
        # Convert string epoch to a readable datetime object
        dt_object = datetime.fromtimestamp(int(raw_ts))
        return dt_object
    except (ValueError, IndexError):
        # Fallback if the filename doesn't follow the pattern
        return datetime.now()
storage_options = {
    "AWS_ACCESS_KEY_ID": access_key,
    "AWS_SECRET_ACCESS_KEY": secret_key,
    "AWS_ENDPOINT_URL": endpoint,
    "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
    "AWS_S3_ADDRESSING_STYLE": "path",
    "AWS_ALLOW_HTTP": "true",
    "region": "us-east-1"
}

In [51]:
def process_json(bucket, prefix):
    paginator = s3.get_paginator("list_objects_v2")

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            src_key = obj["Key"]

            # 1. Skip directories and empty files
            if src_key.endswith("/") or obj['Size'] == 0:
                continue

            print(f"Processing: {src_key}")

            # 2. Read and Parse
            response = s3.get_object(Bucket=bucket, Key=src_key)
            content = response['Body'].read().decode('utf-8')
            data = json.loads(content)

            # 3. Extract Deep Metadata
            keys_list, level = get_deep_keys(data)

            # 4. Correct Record Counting (Flattening for the count)
            # This ensures [[{}, {}]] returns 2, not 1
            temp_data = data
            for _ in range(level):
                if isinstance(temp_data, list) and len(temp_data) > 0:
                    temp_data = [item for sublist in temp_data for item in (sublist if isinstance(sublist, list) else [sublist])]
            record_count = len(temp_data)

            # 5. Build the Metadata Blob (The "Table inside a Table")
            # This blob changes structure based on file type
            metadata_blob = {
                "nesting_level": level,
                "schema_keys": keys_list,
                "file_size_bytes": obj['Size']
            }

            # 6. Prepare Final Catalog Row
            filename = os.path.basename(src_key)
            metadata_row = pd.DataFrame([{
                "file_id": filename,
                "source_type": src_key.split('/')[2],
                "file_type": "JSON",
                "event_time": extract_timestamp_from_filename(filename),
                "record_count": record_count,
                "metadata_blob": metadata_blob,  # The flexible packet
                "processed_at": pd.Timestamp.now()
            }])

            # 7. Append to Master Catalog
            write_deltalake(
                "s3://landing-zone/persistent-landing/structured/file_catalog/",
                metadata_row,
                mode="append",
                schema_mode="merge",
                storage_options=storage_options
            )

In [53]:
process_json("landing-zone", base_uri + "airquality-barcelona/")


Processing: persistent-landing/semistructured/airquality-barcelona/airquality_1773853034.json
Processing: persistent-landing/semistructured/airquality-barcelona/airquality_1773853094.json
Processing: persistent-landing/semistructured/airquality-barcelona/airquality_1773853195.json
Processing: persistent-landing/semistructured/airquality-barcelona/airquality_1773853805.json
Processing: persistent-landing/semistructured/airquality-barcelona/airquality_1773854416.json


In [61]:
# Connect to DuckDB and configure S3 secret for MinIO
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
CREATE OR REPLACE SECRET secret (
    TYPE s3,
    PROVIDER config,
    ENDPOINT '{endpoint.replace("http://", "").replace("https://", "")}',
    KEY_ID '{access_key}',
    SECRET '{secret_key}',
    URL_STYLE 'path',
    USE_SSL false
);
""")

In [62]:
query = "SELECT * FROM delta_scan('s3://landing-zone/persistent-landing/structured/file_catalog/')"
df_view = con.execute(query).df()
display(df_view)

,file_id,source_type,file_type,event_time,record_count,metadata_blob,processed_at
0,airquality_1773854416.json,airquality-barcelona,JSON,2026-03-18 17:20:16,30,"{'file_size_bytes': 51630, 'nesting_level': 2,...",2026-03-19 14:03:02.730930
1,airquality_1773853805.json,airquality-barcelona,JSON,2026-03-18 17:10:05,30,"{'file_size_bytes': 51630, 'nesting_level': 2,...",2026-03-19 14:03:02.357290
2,airquality_1773853195.json,airquality-barcelona,JSON,2026-03-18 16:59:55,30,"{'file_size_bytes': 51630, 'nesting_level': 2,...",2026-03-19 14:03:01.912653
3,airquality_1773853094.json,airquality-barcelona,JSON,2026-03-18 16:58:14,30,"{'file_size_bytes': 51630, 'nesting_level': 2,...",2026-03-19 14:03:01.451288
4,airquality_1773853034.json,airquality-barcelona,JSON,2026-03-18 16:57:14,30,"{'file_size_bytes': 51630, 'nesting_level': 2,...",2026-03-19 14:03:00.663216
5,weather_1773854416.json,weather-barcelona,JSON,2026-03-18 17:20:16,10,"{'file_size_bytes': 1390, 'nesting_level': 1, ...",2026-03-19 14:02:55.973986
6,weather_1773853805.json,weather-barcelona,JSON,2026-03-18 17:10:05,10,"{'file_size_bytes': 1390, 'nesting_level': 1, ...",2026-03-19 14:02:55.747043
7,weather_1773853195.json,weather-barcelona,JSON,2026-03-18 16:59:55,10,"{'file_size_bytes': 1390, 'nesting_level': 1, ...",2026-03-19 14:02:55.487048
8,weather_1773853094.json,weather-barcelona,JSON,2026-03-18 16:58:14,10,"{'file_size_bytes': 1390, 'nesting_level': 1, ...",2026-03-19 14:02:55.236672
9,weather_1773853034.json,weather-barcelona,JSON,2026-03-18 16:57:14,10,"{'file_size_bytes': 1390, 'nesting_level': 1, ...",2026-03-19 14:02:54.859289
